# 02 · Prompting progresivo: de un prompt suelto a conocimiento integrado

**Caso:** clasificar tickets de soporte de telecomunicaciones en `billing`, `technical`, `cancellation` u `other`. El formato de salida siempre será JSON para poder evaluar.

La ruta: empezamos con un prompt sin estructura, vemos por qué falla, le damos estructura y después aplicamos técnicas encima de esa base.

## 0. Preparación

Misma configuración que el notebook 01: llaves en `.env`, LM Studio activo.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from receipt_validation.config import load_settings

settings = load_settings(ROOT)
print(f"Project root: {ROOT}")

Definimos el caso y dos helpers: `call_model` hace la llamada al modelo local y `show` presenta resultados legibles.

In [ ]:
import requests
from IPython.display import Markdown, display

LOCAL_URL = f"{settings['lmstudio_base_url'].rstrip('/')}/chat/completions"
LOCAL_MODEL = settings["default_lmstudio_model"]

TICKET = "Desde ayer no tengo señal. Reinicié el módem dos veces y la luz LOS sigue roja."
CATEGORIES = ["billing", "technical", "cancellation", "other"]

def call_model(prompt, temperature=0.0, max_tokens=1024):
    payload = {
        "model": LOCAL_MODEL,
        "messages": [{"role": "user", "content": prompt}],
        "temperature": temperature,
        "max_tokens": max_tokens,
    }
    response = requests.post(LOCAL_URL, json=payload, timeout=120)
    response.raise_for_status()
    message = response.json()["choices"][0]["message"]
    return message.get("content") or message.get("reasoning_content") or ""

def show(title, body):
    display(Markdown(f"#### {title}\n\n{body}\n\n---"))

## 1. Prompt sin estructura

Primero lo que haría cualquiera: pedirle al modelo que clasifique, sin más. Observen la salida: formato libre, categorías inventadas, imposible de evaluar en automático.

In [ ]:
# TODO: Escribe el prompt "ingenuo": una sola pregunta libre sobre el TICKET, sin estructura.
# Muéstralo con show() y llama call_model para ver la salida en formato libre.
naive_prompt = ""

## 2. El mismo prompt, con estructura

Cinco piezas: rol, categorías cerradas, regla de ambigüedad, entrada delimitada y contrato de salida en JSON. La tarea no cambió; cambió que ahora la salida es predecible y evaluable.

In [ ]:
# TODO: Escribe el prompt estructurado con las cinco piezas:
#   rol, categorías cerradas (CATEGORIES), regla de ambigüedad,
#   entrada delimitada (<ticket>...</ticket>) y contrato de salida JSON
#   {"category": "...", "confidence": 0.0, "reason": "one short sentence"}.
structured_prompt = ""

## 3. Estabilidad: ambos prompts a distintas temperaturas

Corremos las dos versiones a tres temperaturas. El prompt sin estructura se degrada al subir la temperatura; el estructurado mantiene el formato incluso con alta diversidad. La estructura es lo que hace al sistema robusto, no la temperatura baja.

In [ ]:
# TODO: Corre naive_prompt y structured_prompt a temperaturas 0.0, 0.7 y 1.2
# y compara: ¿cuál mantiene el formato al subir la temperatura?
prompts = {"naive": naive_prompt, "structured": structured_prompt}

## 4. One-shot

Sobre la base estructurada añadimos un ejemplo representativo. El ejemplo enseña a la vez la decisión y el formato; conviene que sea correcto, breve y cercano al borde que queremos aclarar.

In [ ]:
# TODO: Agrega UN ejemplo etiquetado (ticket de facturación) antes de pedir
# la clasificación del TICKET. El ejemplo enseña decisión y formato a la vez.
one_shot_prompt = ""

## 5. Few-shot

Varios ejemplos cubren clases y casos límite. No se trata de llenar contexto: cada ejemplo debe aportar una regla observable. Ojo con el orden y con sobrerrepresentar una clase.

In [ ]:
# TODO 1: Crea 4 ejemplos balanceados, uno por categoría (billing, technical,
#   cancellation, other), como pares (texto, etiqueta).
# TODO 2: Formatéalos como <ticket>...</ticket> + JSON y construye few_shot_prompt.
examples = []
few_shot_prompt = ""

## 6. Chain-of-Thought (CoT)

La técnica clásica pide razonar paso a paso antes de responder. En producción suele ser mejor pedir una **justificación breve y verificable** con criterios observables: síntoma, evidencia y decisión.

In [ ]:
# TODO: Prompt con criterios observables en orden: síntoma explícito,
# evidencia ligada a una categoría, resolución de ambigüedad con la lista cerrada.
# Pide JSON con category, confidence y una justificación breve basada en evidencia.
cot_prompt = ""

## 7. Knowledge Generation & Knowledge Integration

Dos llamadas: primero generamos conocimiento del dominio, después lo integramos como contexto controlado para decidir. Esto permite inspeccionar, filtrar o sustituir el conocimiento antes de clasificar.

**Antecesor:** *Generated Knowledge Prompting* generaba conocimiento y lo añadía al prompt final. Separar las dos etapas mejora el control.

In [ ]:
# TODO: Primera llamada — pide hasta 4 reglas de dominio útiles para clasificar
# el TICKET, SIN permitir que elija la categoría final.
knowledge_prompt = ""
generated_knowledge = ""

Segunda etapa: inyectamos el conocimiento revisado y pedimos evidencia más un campo `used_knowledge` para auditar qué se usó.

In [ ]:
# TODO: Segunda llamada — inyecta el conocimiento revisado en <knowledge> y pide
# JSON con category, confidence, evidence y used_knowledge para auditar qué se usó.
integration_prompt = ""

## 8. Salida estructurada garantizada (Pydantic + JSON Schema)

Hasta ahora el JSON se pedía «por favor» dentro del prompt: el modelo puede romper el contrato en cualquier momento. La forma robusta es declararlo con un modelo de Pydantic y pasarlo como `response_json_schema`: el proveedor fuerza la salida a cumplir el esquema y la respuesta se valida de regreso a un objeto tipado. Lo aplicamos sobre nuestra mejor técnica: el prompt few-shot.

In [ ]:
from typing import Literal
from google import genai
from pydantic import BaseModel, Field

client = genai.Client(api_key=settings["google_api_key"])

class TicketClassification(BaseModel):
    # TODO: Declara los tres campos del contrato:
    #   category: Literal con las cuatro categorías cerradas
    #   confidence: float de 0.0 a 1.0
    #   reason: str con una oración corta basada en evidencia
    pass

In [ ]:
# TODO 1: Llama client.models.generate_content con few_shot_prompt y config que fuerce
#   response_mime_type "application/json" y response_json_schema del modelo Pydantic.
# TODO 2: Valida la respuesta con TicketClassification.model_validate_json y muéstrala.

## 9. Cierre

- Zero-shot estructurado es la línea base; suele bastar para tareas claras.
- One/few-shot ayudan cuando la frontera entre clases necesita ejemplos.
- CoT o criterios estructurados sirven cuando la decisión requiere composición.
- Knowledge Generation & Integration sirve cuando hace falta conocimiento inspeccionable.
- Una técnica más compleja solo se conserva si mejora una métrica relevante.
- Cuando el formato es crítico no se pide: se garantiza con `response_json_schema`.

<details><summary><strong>Pausa y conclusión</strong></summary>

1. ¿Qué cambió entre el prompt suelto y el estructurado?
2. ¿Cuál de los dos resistió mejor la temperatura alta y por qué?
3. ¿Cuándo justificarían pagar dos llamadas (knowledge + integration)?
</details>